# 19. 데이터 사전 분할 (Drive-Level 1:1 Balanced Subsets)

이 노트북은 피처 엔지니어링이 완료된 학습 데이터(`train.parquet`)를 로드하여 여러 개의 1:1 균등 서브셋(Subset)으로 분할합니다.
모든 고장 개체와 동일한 개수의 랜덤 정상 개체 데이터를 매칭하며, 개체(serial_number) 단위 시계열을 완전하게 보존합니다.
특히, **서브셋 간 정상 개체가 서로 겹치지 않도록 전체 정상 풀에서 비복원 추출(Disjoint)**을 수행합니다.
생성된 데이터는 `data2/06b_subset_generation/seed_{SEED}/` 경로에 저장되며, 학습 시 자동으로 인식됩니다.

In [ ]:
import sys, os
import pandas as pd
import numpy as np
from pathlib import Path

# ── [독립 설정 구역] ────────────────────────────────────────
# 학습 컨피그와 완전히 독립적으로 실행되도록 경로와 설정을 로컬에 직접 지정합니다.

DATA_ROOT   = "../data2"      # 데이터 루트 폴더 (data 또는 data2)
TRAIN_PATH  = f"{DATA_ROOT}/03_splitting/train.parquet"

TARGET_SEED = 42      # 생성할 데이터셋의 시드 번호
N_SUBSETS   = 10      # 생성할 서브셋 개수

print(f"🚀 독립 분할 작업 준비: Seed={TARGET_SEED}, Subsets={N_SUBSETS}")
print(f"📂 입력 경로: {TRAIN_PATH}")

## 1. 데이터 로드
피처 엔지니어링이 완료된 원본 데이터(`train.parquet`)를 로드합니다.

In [ ]:
print(f"📂 원본 데이터 로딩 중... ({TRAIN_PATH})")
df_train = pd.read_parquet(TRAIN_PATH)
print(f"✅ 로드 완료: {len(df_train):,} rows")

print("⚠️ 학습 단계를 위한 고장 당일(D-DAY) 행 제외 작업 진행 중...")
failed_serials = df_train[df_train['failure'] == 1]['serial_number'].unique()
df_failed = df_train[df_train['serial_number'].isin(failed_serials)]
max_dates = df_failed.groupby('serial_number')['date'].max().reset_index()
max_dates['is_dday'] = True

df_train = df_train.merge(max_dates, on=['serial_number', 'date'], how='left')
df_train = df_train[df_train['is_dday'].isna()].drop(columns=['is_dday'])
print(f"✅ 고장 당일 행 제외 완료! (제외된 행 수: {len(max_dates):,}, 최종 학습 행 수: {len(df_train):,} rows")

## 2. 개체 단위 1:1 서브셋 분할 실행

In [ ]:
print("🔧 개체 단위(serial_number) 1:1 균등 서브셋 분할 중 (서브셋 간 정상 개체 중복 배제)...")

# 1. 고장 개체와 정상 개체의 serial_number 목록 추출
# 고장이 단 한 번이라도 발생한 개체는 고장 개체, 그렇지 않으면 정상 개체로 분류
failed_serials = df_train[df_train['failure'] == 1]['serial_number'].unique()
all_serials = df_train['serial_number'].unique()
healthy_serials = np.array(list(set(all_serials) - set(failed_serials)))

n_failed = len(failed_serials)
n_healthy = len(healthy_serials)

print(f"  * 전체 유니크 개체 수: {len(all_serials):,}개")
print(f"  * 고장 개체 수: {n_failed:,}개")
print(f"  * 정상 개체 수: {n_healthy:,}개")

# 2. 10개 서브셋에 배정할 전체 정상 개체를 비복원 추출 (중복 배제)
total_needed_healthy = n_failed * N_SUBSETS
assert n_healthy >= total_needed_healthy, f"오류: 전체 정상 개체 수({n_healthy}개)가 필요한 수({total_needed_healthy}개)보다 적습니다."

rng_pool = np.random.default_rng(TARGET_SEED)
all_sampled_healthy = rng_pool.choice(healthy_serials, size=total_needed_healthy, replace=False)

print(f"  * 전체 서브셋에서 사용할 총 정상 개체 수: {total_needed_healthy:,}개")

subsets = []
for i in range(N_SUBSETS):
    # 각 서브셋에 할당할 정상 개체 슬라이싱
    sampled_healthy = all_sampled_healthy[i * n_failed : (i + 1) * n_failed]
    
    # 선택된 고장 개체 + 슬라이싱된 정상 개체 병합
    selected_serials = set(failed_serials).union(set(sampled_healthy))
    
    # 전체 데이터에서 해당 개체들의 시계열 로우 추출
    df_sub = df_train[df_train['serial_number'].isin(selected_serials)].copy()
    
    # 학습 시 쏠림 방지를 위해 로우 셔플링
    df_sub = df_sub.sample(frac=1, random_state=TARGET_SEED + i).reset_index(drop=True)
    subsets.append(df_sub)
    print(f"  -> 서브셋 {i+1}/{N_SUBSETS} 완료 (행 수: {len(df_sub):,} rows)")

print(f"\n✅ 총 {len(subsets)}개의 개체 단위 1:1 서브셋(상호 중복 없는 정상 데이터) 생성 완료")

## 3. 파일 저장
결과 폴더에 서브셋을 저장합니다.

In [ ]:
# 상대 경로를 절대 경로로 변환하여 안전하게 저장
save_dir = (Path(DATA_ROOT) / "06_subset_generation" / f"seed_{TARGET_SEED}").resolve()
save_dir.mkdir(parents=True, exist_ok=True)

print(f"💾 저장 경로: {save_dir}")

for i, sub in enumerate(subsets):
    path = save_dir / f"subset_{i}.parquet"
    sub.to_parquet(path, index=False)
    print(f"  [Done] Subset {i+1}/{len(subsets)}: {len(sub):,} rows")

print(f"\n✨ Seed {TARGET_SEED} 버전의 데이터셋 생성이 완료되었습니다!")

## 4. 서브셋 생성 결과 및 개체 단위 1:1 비율 무결성 검증 테스트 (Verification Tests)

생성된 10개 서브셋 파일들의 실제 물리 생성 여부, 서브셋 내 고장 개체와 정상 개체의 1:1 정합성, 그리고 정상 개체로 분류된 드라이브에 고장 기록(failure=1)이 존재하지 않는지를 수학적으로 엄밀히 입증합니다.

In [ ]:
print("🔍 [6-B단계 무결성 검증] 시작...")

try:
    # 1. 파일 개수 및 실제 물리 파일 존재 여부 검증
    print("Test 1: 서브셋 물리 파일 생성 테스트")
    assert len(subsets) == N_SUBSETS, f"오류: 생성된 서브셋 개수({len(subsets)}개)가 설정치({N_SUBSETS}개)와 다릅니다."
    for i in range(N_SUBSETS):
        path = save_dir / f"subset_{i}.parquet"
        assert path.is_file(), f"오류: 서브셋 파일 {path.name}이 누락되었습니다."
    print(f"  -> [PASS] {N_SUBSETS}개 서브셋 파일 정상 저장 확인.")

    # 2. 개체 단위 1:1 비율 정합성 검증
    print("Test 2: 서브셋별 고장 개체 대비 정상 개체 1:1 비율 검증")
    sub_healthy_list = []
    for i in range(N_SUBSETS):
        path = save_dir / f"subset_{i}.parquet"
        df_sub = pd.read_parquet(path)
        
        # 고장 개체: failure=1을 기록한 적이 있는 개체
        sub_failed = df_sub[df_sub['failure'] == 1]['serial_number'].unique()
        # 정상 개체: failure=1을 기록한 적이 없는 개체
        sub_healthy = np.array(list(set(df_sub['serial_number'].unique()) - set(sub_failed)))
        
        sub_healthy_list.append(set(sub_healthy))
        len_failed = len(sub_failed)
        len_healthy = len(sub_healthy)
        
        assert len_failed == len_healthy, (
            f"오류: Subset {i}의 고장 개체 수({len_failed}개)와 "
            f"정상 개체 수({len_healthy}개)가 일치하지 않습니다!"
        )
    print("  -> [PASS] 모든 서브셋 내 고장/정상 개체 비율 1:1 완벽 일치.")

    # 3. 정상 개체의 순수성 검증
    print("Test 3: 정상 개체 타임라인 내 고장 기록(failure=1) 부존재 검증")
    for i in range(N_SUBSETS):
        path = save_dir / f"subset_{i}.parquet"
        df_sub = pd.read_parquet(path)
        
        sub_failed = set(df_sub[df_sub['failure'] == 1]['serial_number'].unique())
        
        # 정상 개체들의 모든 행에서 failure가 0인지 검사
        df_healthy_rows = df_sub[~df_sub['serial_number'].isin(sub_failed)]
        assert (df_healthy_rows['failure'] == 0).all(), f"오류: Subset {i}의 정상 개체 데이터 중 failure=1인 행이 존재합니다!"
    print("  -> [PASS] 정상 개체의 고장 기록(failure=1) 부존재 확인.")

    # 4. 서브셋 간 정상 개체 중복 여부(비복원 추출) 검증
    print("Test 4: 서브셋 간 정상 개체(Healthy Drive) 중복 누수 배제 검증")
    for i in range(N_SUBSETS):
        for j in range(i + 1, N_SUBSETS):
            overlap = sub_healthy_list[i].intersection(sub_healthy_list[j])
            assert len(overlap) == 0, (
                f"오류: Subset {i}와 Subset {j} 사이에 중복되는 정상 개체가 {len(overlap)}개 존재합니다!"
            )
    print("  -> [PASS] 모든 서브셋 간 정상 개체 교집합 0% (완전한 비복원 추출 완료). ")

    print("\n🏆 [6-B단계 정합성 검증 완료] 모든 엄격한 테스트 조건을 만족합니다!")
finally:
    pass